# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Our lane (the **Bleed Tracker**) is a **scoring / ranking task, implemented as a
binary classifier whose predicted probability becomes the priority score**. The decision it
serves — "which top-tier pages must an editor triage first?" — is literally "which ones
first?", which maps to *ranking / scoring* on the framing table, not plain classification: the
editor never sees a bare yes/no, they see a capped, ordered queue (top 50) they work down until
capacity runs out. We get that ordering by training a classifier on `is_declining_label` and
sorting pages by predicted decline probability — classification supplies the score, ranking is
how it's consumed. Clustering is wrong here because we already have a named outcome to predict,
not an unlabeled "what groups exist" question. Pure signal analysis is wrong because the
customer (the editor) needs a decision-ready list, not an effect-size table.

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Our lane's slice: active pages old enough to have a full 90d trailing window
active = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

print("Task shape in code terms: predict_proba() -> sort descending -> take head(50)")
print(f"Rows in scope: {len(active):,}")
print("trend_direction value counts (source of our current label):")
print(active["trend_direction"].value_counts())


Task shape in code terms: predict_proba() -> sort descending -> take head(50)
Rows in scope: 30,000
trend_direction value counts (source of our current label):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

We predict `is_declining_label` (1 when `trend_direction == "down"`, i.e. impressions in the
last 30 days fell more than 20% versus the prior 30 days). We are naming this honestly as a
**proxy, not a clean observed future outcome**: `trend_direction` is computed from `trend_pct`,
which compares two windows that both sit *inside* the same trailing-90-day snapshot our other
features are aggregated over — it is a rule applied to already-elapsed history, not a label
measured in a genuinely future window after our feature cutoff. Per the data dictionary, `trend_direction` and `trend_pct` are excluded from the
feature set for exactly this reason — using them as inputs would leak the label into itself.
The fix is scoring pages as of month *t* and checking `fact_content_daily_performance` for
an actual decline in month *t+1* (this needs the warehouse's forward-looking daily fact table). For this week's
framing, we proceed with the proxy but it is provisional, not final.

In [7]:
active["is_declining_label"] = (active["trend_direction"] == "down").astype(int)

print(f"Label base rate: {active['is_declining_label'].mean():.4%}")
print(f"Positives: {active['is_declining_label'].sum():,} / {len(active):,}")

# Confirm the two leakage-source columns exist and are the ones we exclude from features
leak_cols = ["trend_direction", "trend_pct"]
print(f"Excluded-from-features columns present: {[c for c in leak_cols if c in active.columns]}")


Label base rate: 54.2067%
Positives: 16,262 / 30,000
Excluded-from-features columns present: ['trend_direction', 'trend_pct']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** (carried over from w01): of the top 50 pages our score ranks highest-risk,
what fraction are actually labeled declining? Editorial capacity is fixed and small relative to
30,000 pages, so a metric like ROC-AUC — which rewards correct ordering across *all* ranks —
overstates what matters. Only the head of the queue gets acted on. "Good" means precision@50
that clears the base rate by a wide, defensible margin: the base rate below (~54%) is what a
person picking 50 pages at random would already get right, so our bar for "the model earns its
keep" is precision@50 noticeably above that number — and, once we have a genuinely forward
label, above what today's best single-signal heuristic can reach.

In [8]:
def precision_at_k(scored_df, score_col, label_col="is_declining_label", k=50, ascending=False):
    """Sort by score_col (most-at-risk first) and report the label hit-rate in the top k."""
    ranked = scored_df.sort_values(score_col, ascending=ascending)
    top_k = ranked.head(k)
    return top_k[label_col].mean()

base_rate = active["is_declining_label"].mean()
print(f"Base rate (random 50 pages): {base_rate:.2%}")
print(f"Target: precision@50 must clear {base_rate:.2%} by a defensible margin to justify the model.")


Base rate (random 50 pages): 54.21%
Target: precision@50 must clear 54.21% by a defensible margin to justify the model.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one content item (page), for one client, summarized over its trailing 90-day
window** — the same grain as the raw CSV. Our lane's slice filters that down to *active* pages
(`impressions_90d > 0` and `content_age_days >= 90`, so every page has both real search
traffic and a full 90 days of history to compute the trend windows from). `content_id` is the
row identity; `client_id` groups pages that belong to the same client, which matters for our
train/test split later (grouped by client, never a feature).

In [9]:
unit_cols = ["content_id", "client_id", "content_type", "impressions_90d",
             "clicks_90d", "trend_direction", "is_declining_label"]
display_cols = active[unit_cols].head(5)
print(display_cols)

# Grain probe: one row per content_id, no duplicates
dupes = active.groupby("content_id").size()
print(f"\ncontent_id is unique per row: {(dupes.max() == 1)} (max rows per id: {dupes.max()})")
print(f"Distinct clients represented: {active['client_id'].nunique()}")


             content_id          client_id     content_type  impressions_90d  \
0  content_304f48230142  client_f369cb89fc  keyword article             3803   
1  content_a1fb4e703a9e  client_4e07408562  keyword article            15320   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article            12581   
3  content_331d6c4de07b  client_19581e27de  keyword article            11751   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article            19140   

   clicks_90d trend_direction  is_declining_label  
0          29            down                   1  
1           7            down                   1  
2          11            down                   1  
3          58          stable                   0  
4          24            down                   1  

content_id is unique per row: True (max rows per id: 1)
Distinct clients represented: 32


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

We tested the obvious if-statement rules a person would reach for first: flag the 50 pages with
the lowest `scroll_rate`, or the lowest `ctr`, or the lowest `engagement_rate`, as "declining."
Below, the single-signal rules land at 50–58% precision@50 — at or *below* the 54% base rate, so
picking on one engagement metric alone is no better than picking 50 pages at random. Even a
hand-combined average of all three only reaches 62%. The correlation table makes the reason
legible: each engagement signal has almost **zero linear correlation with the label** (all
under |0.07|) — none of them is individually informative. That's the classic "too messy for an
if-statement" case. The real signal, if it exists, is a tangled
combination across many of the 44 columns (traffic volume, position, keyword competition,
content age, freshness, content type) rather than one obvious threshold — exactly the setting
where a multivariate model can find structure a hand-written rule can't.

In [10]:
signal_cols = ["scroll_rate", "ctr", "engagement_rate"]
sub = active.dropna(subset=signal_cols).copy()

# Single-signal "if-statement" rules: flag the 50 pages lowest on each signal
for col in signal_cols:
    p = precision_at_k(sub, col, k=50, ascending=True)
    print(f"precision@50, lowest {col:>16s}: {p:.2%}")

# A hand-combined rule: average rank across all three signals
for col in signal_cols:
    sub[f"rank_{col}"] = sub[col].rank(ascending=True)
sub["combo_rank"] = sub[[f"rank_{c}" for c in signal_cols]].mean(axis=1)
p_combo = precision_at_k(sub, "combo_rank", k=50, ascending=True)
print(f"precision@50, hand-combined rule:      {p_combo:.2%}")
print(f"base rate for reference:               {base_rate:.2%}")

print("\nCorrelation of each signal with the label (near-zero => no single if-statement works):")
print(sub[signal_cols + ["is_declining_label"]].corr()["is_declining_label"])


precision@50, lowest      scroll_rate: 58.00%
precision@50, lowest              ctr: 50.00%
precision@50, lowest  engagement_rate: 54.00%
precision@50, hand-combined rule:      62.00%
base rate for reference:               54.21%

Correlation of each signal with the label (near-zero => no single if-statement works):
scroll_rate          -0.002958
ctr                  -0.062082
engagement_rate      -0.012481
is_declining_label    1.000000
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.